# 简单机器人


In [4]:
import os
from typing import TypedDict, List

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

load_dotenv()


True

In [5]:
class AgentState(TypedDict):
    messages: List[HumanMessage]

## 使用 OpenAI 中转接口


In [8]:
llm = ChatOpenAI(
    model="gpt-4o-mini",  # 也可以换成你的中转服务支持的其它 OpenAI 模型
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL", "https://api.openai-proxy.org/v1"),
)

# 简单测试：确认代理地址和 key 配置无误
# response = llm.invoke([HumanMessage(content="你好，请用一句话介绍 LangGraph")])
# print(response.content)


def process(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    print(f"LLM response: {response.content}")
    return state

In [10]:
graph = StateGraph(AgentState)
graph.add_node("process", process)
graph.add_edge(START, "process")
graph.add_edge("process", END)
agent = graph.compile()

In [ ]:
user_input = input("Enter:")
while user_input != "exit":
    agent.invoke({"messages": [HumanMessage(content=user_input)]})
    user_input = input("Enter:")

LLM response: Hello! How can I assist you today?
LLM response: 抱歉，我无法知道您的名字。如果您愿意，可以告诉我您的名字。
LLM response: 你好，Lily！很高兴见到你。有什么我可以帮助你的吗？
LLM response: 抱歉，我无法知道您的名字。请告诉我您的名字，或者有什么我可以帮助您的吗？
